# Lesson 5.4 — Discovery and Communicating Trade-offs

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Run structured discovery to gather requirements and constraints
- Translate technical trade-offs into stakeholder language
- Produce a decision brief that drives alignment

**You will produce:** A discovery questionnaire and a stakeholder trade-off brief.

## Concepts

**Discovery** is the structured gathering of the problem, constraints, success metrics, data, risks, and stakeholders *before* designing. Skipping it is the root of most failed AI projects. A good discovery questionnaire covers: business goal and KPI, users and volumes, data sources and sensitivity, latency/accuracy/cost targets (SLAs), compliance context, and definition of done.

**Communicating trade-offs** is a core architect skill (Domain 6). Stakeholders decide in terms of *value, cost, risk, and time* — not tokens and tiers. The architect presents options as a small set of named choices with their consequences, makes a recommendation, and states what each option gives up. This builds on the ADR (Lesson 1.5) but is pitched to executives, not engineers.

> **Reference:** [Anthropic — Solution design & best practices](https://docs.anthropic.com/en/docs/overview)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Create a structured discovery questionnaire

A reusable questionnaire ensures no critical requirement is missed. We group questions by theme and persist them.

In [ ]:
discovery = {
  "business": ["What decision/outcome improves? What KPI moves?","What is the cost of the status quo?"],
  "users":    ["Who uses it and at what volume?","What is the required SLA (latency, uptime)?"],
  "data":     ["What sources ground answers?","What sensitivity/regulatory class (PII/PHI)?"],
  "quality":  ["How is a good answer defined/measured?","What accuracy floor is acceptable?"],
  "risk":     ["What is the worst-case failure? Reversible?","Where must a human approve?"],
  "done":     ["What is the definition of done for v1?","Who signs off?"],
}
with open("discovery_questionnaire.json","w",encoding="utf-8") as fh:
    json.dump(discovery, fh, indent=2)
for theme, qs in discovery.items():
    print(theme.upper()); [print('  -',q) for q in qs]

## Code-Along: Step 2 — Draft a stakeholder trade-off brief

We render options as value/cost/risk/time, recommend one, and state the trade-off — the language executives act on.

In [ ]:
options = [
  {"name":"A: Haiku, no RAG","value":"fast, cheap","cost":"$","risk":"lower accuracy","time":"2 wks"},
  {"name":"B: Sonnet + RAG + HITL","value":"accurate, auditable","cost":"$$","risk":"moderate latency","time":"6 wks"},
  {"name":"C: Opus + multi-agent","value":"max accuracy","cost":"$$$$","risk":"cost/latency/complexity","time":"12 wks"},
]
def trade_off_brief(options, recommend):
    lines = ["# Trade-off Brief",""]
    for o in options:
        lines.append(f"- **{o['name']}** — value: {o['value']}; cost: {o['cost']}; risk: {o['risk']}; time: {o['time']}")
    lines += ["", f"## Recommendation\n{recommend}"]
    return "\n".join(lines)

brief = trade_off_brief(options, "Option B: best accuracy/auditability per dollar; meets the 20s SLA with HITL on high-impact cases.")
with open("tradeoff_brief.md","w",encoding="utf-8") as fh: fh.write(brief)
print(brief)

## Exercise

Use the discovery questionnaire on a *new* scenario (e.g. an internal HR policy assistant). Fill answers, then generate a 3-option trade-off brief with a recommendation. The grader is whether a non-technical VP could choose from your brief without asking what a token is.

In [ ]:
# TODO: answer the discovery questionnaire for your scenario
# TODO: build 3 options (value/cost/risk/time) and a recommendation brief

## Key Takeaways

- Discovery before design: capture goal, users, data, SLAs, risk, and definition of done.
- Stakeholders decide in value/cost/risk/time — translate tokens and tiers into those.
- Present a small set of named options, recommend one, and state what it gives up.
- A good brief lets a non-technical decision-maker choose confidently.

---

**Next:** [Lesson 5.5 — Architecture Documentation and Lifecycle Management](./lesson-5.5-architecture-docs-lifecycle.ipynb)